## ⚙️ 设备管理与上下文控制


在多卡（多 GPU）环境下，合理地指定和切换 GPU 是深度学习开发的基本功。PyTorch 提供了多种方式来管理当前操作的 GPU，下面我为你详细解释这三个核心方法，并配上实用的代码示例：

1. torch.cuda.current_device()：查看当前在用哪张卡
- 这个方法会返回当前线程正在使用的 GPU 索引（从 0 开始）。如果你没有手动指定过，PyTorch 默认会使用 cuda:0（第一张显卡）。
2. torch.cuda.set_device(device)：<font color='red'>全局切换默认显卡</font>
- 这是一个全局设置。一旦调用它，当前进程中后续所有未明确指定设备的张量（Tensor）和模型，都会默认被分配到这张显卡上。它就像是一个“总开关”。
3. with torch.cuda.device(device)：<font color='red'>临时切换</font>显卡的上下文管理器
- 这是一个局部、临时的设置。它通常配合 with 语句使用，只在 with 代码块内部临时切换默认显卡，一旦离开这个代码块，默认设备会自动恢复成之前的状态。


In [ ]:
import torch

# 假设你的机器上有至少 2 张显卡 (cuda:0 和 cuda:1)

# 1. 查看初始状态
print(f"1. 初始默认显卡索引: {torch.cuda.current_device()}") 
# 预期输出：0 (默认通常是第一张卡)

# 2. 使用 set_device 全局切换到 1 号卡
torch.cuda.set_device(1)
print(f"2. 全局切换后，当前显卡索引: {torch.cuda.current_device()}")
# 预期输出：1

# 在这个全局设置下，创建的张量会自动放在 cuda:1 上
x = torch.randn(2, 2).cuda()
print(f"   张量 x 所在的设备: {x.device}") 
# 预期输出：cuda:1

# 3. 使用上下文管理器临时切换到 0 号卡
with torch.cuda.device(0):
    print(f"   3. 进入 with 块后，当前显卡索引: {torch.cuda.current_device()}")
    # 预期输出：0
    
    # 在这个代码块内部，创建的张量会自动放在 cuda:0 上
    y = torch.randn(2, 2).cuda()
    print(f"      张量 y 所在的设备: {y.device}")
    # 预期输出：cuda:0

# 4. 离开 with 代码块，默认设备会自动恢复
print(f"4. 离开 with 块后，当前显卡索引: {torch.cuda.current_device()}")
# 预期输出：1 (自动恢复到了之前的全局设置)

# 此时再创建张量，依然会放在 cuda:1 上
z = torch.randn(2, 2).cuda()
print(f"   张量 z 所在的设备: {z.device}")
# 预期输出：cuda:1

## 💡 核心总结与使用建议
- current_device()：通常用于调试，帮你确认当前代码到底跑在哪张卡上。
- set_device()：适合在脚本的最开头使用。比如你明确知道整个训练任务都要跑在 3 号卡上，就在代码最开始写一句 torch.cuda.set_device(3)。
- with torch.cuda.device()：适合在复杂的逻辑中临时借用其他显卡。比如你的主干网络在 0 号卡，但某个特定的预处理或子模块需要临时放到 1 号卡上跑，用 with 语句既安全又不会打乱全局的设备分配。

